# Walkthrough for UAB

In this notebook, I gathered the relevant code from previous notebooks.

## Data preparation

In [4]:
%%capture

import pandas as pd
import numpy as np
import collections
import json
from matplotlib.dates import date2num
from pandas.api.types import is_datetime64_any_dtype as is_datetime


PATH = "../data/preemi-30apr2021.dta"

df = pd.read_stata(PATH)

df[df == ""] = np.nan

columns = list(df)

# Unique columns
df = df.drop(columns=["protocolid", "personid", "pr_id", "ch_id", "mt_id"])

# Event status
df = df.drop(columns=[c for c in columns if "eventstatus_e" in c])

# Versions
df = df.drop(columns=[c for c in columns if "versionname_e" in c])

# Constant or almost unique
df = df.drop(columns=["haemounava_e1"])
df = df.drop(columns=["checklist_e1"])
df = df.drop(columns=["matstata_e2"])

# Remove columns with high NaNs
N = df.shape[0]
for col in columns:
    if col not in df: continue
    n = df[df[col].isnull()].shape[0]
    if n/N > 0.5:
        df = df.drop(columns=[col])

# Remove redundant _e

red_cols = collections.defaultdict(list)
for c in list(df.columns):
    x = c.split("_")
    if len(x[-1]) == 2 and x[-1][0] == "e":
        red_cols["_".join(x[:-1])] += ["_".join(x)]

def get_to_drop(v):
    b = -1
    best = None
    for x in v:
        if int(x[-1]) > b:
            b = int(x[-1])
            best = x
    return [x for x in v if x != best]

for k,v in red_cols.items():
    v = get_to_drop(v)
    #df = df.drop(columns=v)

df = df.drop(columns = ["compperson_e5"])

# Minor cleans

df.loc[df["facid_e5"] == "FO3", "facid_e5"] = "F03"
df.loc[df["facid_e5"] == "FO2", "facid_e5"] = "F02"

# Remove dirty variables

df = df.drop(columns=["BWDQ_cat", "birthweightcat1", "gestagecat2", "gestagecat1"])

# Newly realised redundant
df = df.drop(columns = ["facid_e5"])

# Clean data type

def datatype_cleaner(df, col):
    df_ = df.copy()
    df_ = df_[df_[col].notnull()]
    N = df_.shape[0]
    df_["numnew"] = pd.to_numeric(df_[col], errors="coerce")
    df_["datnew"] = pd.to_datetime(df_[col], errors="coerce")
    if df_[df_["numnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_numeric(df[col])
        return df
    if df_[df_["datnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df.loc[df[col] < pd.Timestamp(1910,1,1)] = np.nan
        return df
    return df

for col in columns:
    if col not in df: continue
    df = datatype_cleaner(df, col)


df.loc[df["matage"] > 60, "matage"] = np.nan
df.loc[df["eduyrs_e1"] > 15, "eduyrs_e1"] = 15
df.loc[df["matwei_e1"] > 200, "matwei_e1"] = np.nan
df.loc[df["babwght_e2"] > 8000, "babwght_e2"] = np.nan
df.loc[df["bwdat"] == "Birth weight missing", "bwdat"] = np.nan
df.loc[df["fetneo_e2"] == "3=Fresh Stillbirth (No movement", "fetneo_e2"] = "3=Fresh Stillbirth"
df[df == "Unknown"] = np.nan
df[df == "unknown"] = np.nan
df[df == "Don't know"] = np.nan

df["vitalstatus_ltf"] = np.nan
df.loc[df["vitalstatus"] == "LTF", ["vitalstatus_ltf"]] = "LTF"
df.loc[(df["vitalstatus"].notnull()) & (df["vitalstatus"] != "LTF"), ["vitalstatus_ltf"]] = "Live birth or Pregnancy loss"
df.loc[df["vitalstatus"] == "LTF", ["vitalstatus"]] = np.nan

df["pr_outcome_miscarriage"] = np.nan
df.loc[df["pr_outcome"] == "Miscarriage", ["pr_outcome_miscarriage"]] = "Miscarriage"
df.loc[(df["pr_outcome"] != "Miscarriage") & (df["pr_outcome"].notnull()), ["pr_outcome_miscarriage"]] = "No miscarriage"
df["pr_outcome_alive"] = np.nan
df.loc[df["pr_outcome"].isin(["Miscarriage", "Stillbirth"]), ["pr_outcome_alive"]] = "Miscarriage or Stillbirth"
df.loc[df["pr_outcome"] == "Live birth", ["pr_outcome_alive"]] = "Live birth"
df = df.drop(columns=["pr_outcome"])

outcomes = {
    "Miscarriage": ["pr_outcome_miscarriage"],
    "Outcome Death": ["pr_outcome_alive"],
    "Early Neonatal Death": ["infsta_e3"],
    "Late Neonatal Death": ["infsta_e4"],
    "Pre-term Delivery": ["preterm", "babterm_e2"] # babterm_e2 is better according to previous tests
}

gestation = {
    "Gestation": ["g_age", "gestage"], # g_age is in days, gestage in weeks
    "Expected Due Date": ["edd_e1", "estedd_e1"], # edd_e1
    "Last Menstrual Period": ["mensdate_e1"], # integer
    "Delivery Date": ["delidate1_e1", "motconpl_e2"], # I am removing motconpl_e2 value (most of them are 0)
    "Method of Determining Gestation": ["gestmethod"] # Method gestation - missing in the majority of cases: remove
}

counfounders = {
    "Maternal Age": ["matage", "age_cat", "matagecat", "dob_day"], # matage
    "School Level": ["schlev_e1"], # categorical
    "Years of Education": ["eduyrs_e1"], # numeric
    "Parity": ["parity_lbsb", "parity_cat", "parity"], # Number of viable births / best variable here is lbsb, counting the actual number
    "Maternal Height": ["mathei_e1"],
    "Maternal Weight": ["matwei_e1"],
    "Antenatal Visits": ["antcarfreq_e2", "anc_visit"], # integer
    "Delivery By": ["delivby_e2"], # Categorical - nurse is the majority
    "Delivery Place": ["delivwhr_e2", "delivfac_e2", "deplace"], # first is hospital, clinic, home, other...
    "Type of Delivery Place": ["delivfac_e2"],
    "Mode of Delivery": ["detype", "mod_e2"], # cesarean, vaginal
    "Baby Sex": ["sex", "babysex_e2"], # male / female for the baby
    "Multiple Birth": ["multbirth_e2", "multiple"],
    "Birthweight": ["birthweight"],
    "Birthweight Measure": ["bwdat"],
}

neonatal_tx = {
    "Neonatal Antibiotics": ["neotreant_e2"], # 0, 1, 2
    "CPAP": ["neotrecpap_e2"], # 0, 1, 2
    "Oxygen": ["neotreoxy_e2"] # 0, 1, 2
}

interventions = {
    "Antenatal Visits": ["antcarfreq_e2"],
    "Dexamethasone": ["mattredex_e2"], # 0, 1, 2
    "Kangaroo Mother Care": ["neotrekmc_e2"], # 0, 1, 2
    "Cord care Chlorhexidine": ["neotremcc_e2"], # 0, 1, 2
    "Bag and Mask Resuscitation": ["babresbm_e2"],
}

all_variables = dict((k,v[0]) for d in [outcomes, gestation, counfounders, neonatal_tx, interventions] for k,v in d.items())

all_columns = list(set([x for k,v in all_variables.items() for x in [v]]))

dt = df[all_columns]

def is_date_column(df, c):
    if is_datetime(df[df[c].notnull()][c]):
        return True
    else:
        return False

def convert_date_to_num(x):
    try:
        return date2num(x)
    except:
        return np.nan
    
dt = dt.copy()
for c in list(dt.columns):
    if is_date_column(dt, c):
        dt.loc[:, [c]] = [convert_date_to_num(x) for x in list(dt[c])]
        
def binarize_variable(dt, c, t):
    b = c + "_bin"
    dt[b] = np.nan
    dt.loc[dt[c] == t, [b]] = 1
    dt.loc[(dt[c] != t) & (dt[c].notnull()), [b]] = 0
    dt = dt.drop(columns = [c])
    dt = dt.rename(columns={b: c})
    return dt

dt = binarize_variable(dt, "preterm", "Preterm")
dt = binarize_variable(dt, "pr_outcome_miscarriage", "Miscarriage")
dt = binarize_variable(dt, "pr_outcome_alive", "Miscarriage or Stillbirth")
dt = binarize_variable(dt, "infsta_e3", "Dead")
dt = binarize_variable(dt, "infsta_e4", "Dead")

dt = dt.sort_values(by="mensdate_e1").copy()
dt = dt[dt["mensdate_e1"].notnull()].reset_index(drop=True)

variable_stages = {
    "Day0": [
        "Maternal Age",
        "School Level",
        "Years of Education",
        "Parity",
        "Maternal Height",
        "Maternal Weight",
        "Multiple Birth",
    ],
    "Pregnancy": [
        "Antenatal Visits"
    ],
    "PreDelivery": [
        "Gestation",
        "Method of Determining Gestation",
        "Last Menstrual Period",
        "Delivery Date",
        "Type of Delivery Place",
        "Delivery Place",
        "Delivery By",
        "Mode of Delivery"
    ],
    "PostDelivery": [
        "Birthweight",
        "Birthweight Measure",
        "Baby Sex",
        "Dexamethasone",
        "CPAP",
        "Oxygen",
        "Kangaroo Mother Care",
        "Cord care Chlorhexidine",
        "Bag and Mask Resuscitation"
    ],
}

modifiable_variables = {
    "Antenatal Visits",
    "Dexamethasone",
    "CPAP",
    "Oxygen",
    "Kangaroo Mother Care Skin to Skin",
    "Cord care Chlorhexidine",
    "Bag and Mask Resuscitation"
}

categories = {
    "Outcomes": outcomes,
    "Gestation": gestation,
    "Counfounders": counfounders,
    "NeonatalTreatment": neonatal_tx,
    "Interventions": interventions
}


with open("../data/processed/variable_stages.json", "w") as f:
    json.dump(variable_stages, f, indent=4)

with open("../data/processed/modifiable_variables.json", "w") as f:
    json.dump(list(modifiable_variables), f, indent=4)

with open("../data/processed/all_variables.json", "w") as f:
    json.dump(all_variables, f, indent=4)

with open("../data/processed/categories.json", "w") as f:
    json.dump(categories, f, indent=4)

dt.to_csv("../data/processed/dt.csv", index=False)

readable_columns = {}
for k,v in categories.items():
    for l,w in v.items():
        readable_columns[w[0]] = l

dr = df[[k for k in readable_columns.keys()]].copy()

dr.rename(columns=readable_columns, inplace=True)

dr = dr[dr["Outcome Death"].notnull()]
dr = dr[dr["Pre-term Delivery"].notnull()]
dr.drop(columns=["Delivery Date"], inplace=True) # half of the entries did not have delivery date...
dr = dr.sort_values(by="Last Menstrual Period").reset_index(drop=True)
dr[dr == 999] = np.nan
dr.to_csv("../data/processed/dr.csv", index=False)

## ML model preparation

In [5]:
labels = ["Pre-term Delivery", "Miscarriage", "Outcome Death"]
d0 = dr[labels + variable_stages["Day0"]]
for l in labels:
    d0 = d0[d0[l].notnull()]
d0.to_csv("../data/processed/ml_d0_dayzero.csv", index=False)

labels = ["Outcome Death"]
predelivery_covariates = ["Antenatal Visits", "Gestation", "Type of Delivery Place", "Delivery Place", "Mode of Delivery"]
covariate_labels = ["Pre-term Delivery"]
d1 = dr[labels + variable_stages["Day0"] + predelivery_covariates + covariate_labels]
d1.to_csv("../data/processed/ml_d1_predelivery.csv", index=False)

labels = ["Early Neonatal Death"]
d2 = dr[labels + variable_stages["Day0"] + predelivery_covariates + variable_stages["PostDelivery"] + covariate_labels]
d2 = d2[dr["Outcome Death"] == "Live birth"]
d2.to_csv("../data/processed/ml_d2_earlydeath.csv", index=False)

labels = ["Late Neonatal Death"]
d3 = dr[labels + variable_stages["Day0"] + predelivery_covariates + variable_stages["PostDelivery"] + covariate_labels]
d3 = d3[dr["Early Neonatal Death"] == "Alive"]
d3.to_csv("../data/processed/ml_d3_latedeath.csv", index=False)

## Exploratory ML with AutoGluon

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

dt_ = dt[dt["pr_outcome_alive"].notnull()]
print(dt_[dt_["pr_outcome_alive"] == 1].shape)
dt_ = dt_[dt_["delidate1_e1"].notnull()]
print(dt_[dt_["pr_outcome_alive"] == 1].shape)
dt_ = dt_[dt_["preterm"] == 0]
print(dt_[dt_["pr_outcome_alive"] == 1].shape)
dt_ = dt_.reset_index(drop=True)

all_outcomes = set([v[0] for k, v in outcomes.items()])

target = "pr_outcome_alive"
columns = [c for c in list(dt_.columns) if c not in all_outcomes] + [target]
allowed = [all_variables[k] for k in variable_stages["Day0"]]
allowed += [all_variables[k] for k in variable_stages["Pregnancy"]]
allowed += [all_variables[k] for k in variable_stages["PreDelivery"]]
#allowed += [all_variables[k] for k in ["Last Menstrual Period", "Delivery Date"]]
columns = [c for c in columns if c in allowed + [target]]
#columns = [c for c in columns if "date" not in c]

dt_ = dt_[columns]
dt_ = dt_[dt_[target].notnull()]

print(dt_.shape)
print(dt_)


dt_ = TabularDataset(dt_)

def train_test_split(df, train_size):
    n = df.shape[0]
    train = df.head(int(n*train_size))
    test = df.tail(int(n*(1-train_size)))
    print(train[train[target] == 1].shape[0])
    print(test[test[target] == 1].shape[0])
    return train, test

dt_tr, dt_te = train_test_split(dt_, 0.7)

In [ ]:
model = TabularPredictor(label=target, problem_type="binary", eval_metric="roc_auc")
model.fit(dt_tr, presets="best_quality")
model.evaluate(dt_te)

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(dt_te[target], model.predict_proba(dt_te)[1])

fig, ax = plt.subplots(1,1, figsize=(5,5))
ax.plot(fpr, tpr)

In [ ]:
model.feature_importance(dt_te)